# Uji statistik berpasangan antar model (paired bootstrap dan McNemar)

In [1]:
import os, json, itertools
import numpy as np
import pandas as pd
from scipy.stats import binomtest

PRED_FILES = {
    "ViT-B/16": "preds/vit_b16_test_preds.csv",
    "Improved Swin": "preds/swin_test_preds.csv",
    "MobileNetV3 Large": "preds/mobilenetv3_test_preds.csv",
    "ResNet50": "preds/resnet50_test_preds.csv",
    "DenseNet121": "preds/densenet121_test_preds.csv",
}
FLAGGED_JSON = "near_dup_report/flagged_test_files.json"
B = 10000
SEED = 42
ALPHA = 0.05
OUT_DIR = "stats_report"
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Muat prediksi dan pastikan semua model dinilai pada gambar yang sama

In [2]:
norm = os.path.normpath
preds = {}
for name, f in PRED_FILES.items():
    d = pd.read_csv(f)
    d["fp"] = d["filepath"].map(norm)
    preds[name] = d

names = list(PRED_FILES)
ref = preds[names[0]]
for n in names[1:]:
    d = preds[n]
    assert len(d) == len(ref), f"{n}: jumlah gambar berbeda"
    assert (d["fp"].values == ref["fp"].values).all(), f"{n}: urutan atau isi gambar test berbeda"
    assert (d["y_true"].values == ref["y_true"].values).all(), f"{n}: label berbeda"

flagged = set(norm(p) for p in json.load(open(FLAGGED_JSON)))
is_flag = ref["fp"].isin(flagged).values
K = int(max(ref["y_true"].max(), *[preds[n]["y_pred"].max() for n in names])) + 1
print("Gambar test asli:", len(ref), "| ditandai:", int(is_flag.sum()),
      "| bersih:", int((~is_flag).sum()), "| kelas:", K)
print("File ditandai yang tidak ditemukan di test:", len(flagged - set(ref["fp"])))

Gambar test asli: 1438 | ditandai: 380 | bersih: 1058 | kelas: 45
File ditandai yang tidak ditemukan di test: 0


## 2. Fungsi metrik dan uji

In [ ]:
def all_metrics(yt, yp, K=K):

    cm = np.bincount(yt * K + yp, minlength=K * K).reshape(K, K)
    tp = np.diag(cm).astype(float)
    pred, true = cm.sum(0).astype(float), cm.sum(1).astype(float)
    ok = (pred + true) > 0
    prec = np.where(pred > 0, tp / np.where(pred > 0, pred, 1), 0.0)
    rec = np.where(true > 0, tp / np.where(true > 0, true, 1), 0.0)
    f1 = np.where(ok, 2 * tp / np.where(ok, pred + true, 1), 0.0)
    return tp.sum() / cm.sum(), prec[ok].mean(), rec[ok].mean(), f1[ok].mean()

METRICS = ("acc", "prec", "rec", "f1")

def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running, m = 0.0, len(p)
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj

def boot_p(d):
    return min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))

def fmt_p(p, B=B):
    return f"< {1/B:.4f}" if p < 1 / B else f"{p:.4f}"

## 3. Analisis

In [4]:
def analyze(mask, label):
    yt = ref["y_true"].values[mask]
    P = {n: preds[n]["y_pred"].values[mask] for n in names}
    n_img = len(yt)
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, n_img, size=(B, n_img))

    boot = {n: {k: np.empty(B) for k in METRICS} for n in names}
    for b in range(B):
        ib = idx[b]
        ytb = yt[ib]
        for n in names:
            vals = all_metrics(ytb, P[n][ib])
            for k, v in zip(METRICS, vals):
                boot[n][k][b] = v

    rows = []
    for n in names:
        point = dict(zip(METRICS, all_metrics(yt, P[n])))
        row = {"himpunan": label, "model": n, "n_gambar": n_img}
        for k in METRICS:
            row[k] = point[k]
            row[k + "_ci_low"] = np.percentile(boot[n][k], 2.5)
            row[k + "_ci_high"] = np.percentile(boot[n][k], 97.5)
        rows.append(row)
    per_model = pd.DataFrame(rows)

    pair_rows = []
    for a, c in itertools.combinations(names, 2):
        pa, pc = dict(zip(METRICS, all_metrics(yt, P[a]))), dict(zip(METRICS, all_metrics(yt, P[c])))
        da = boot[a]["acc"] - boot[c]["acc"]
        df = boot[a]["f1"] - boot[c]["f1"]
        ca, cc = (P[a] == yt), (P[c] == yt)
        b_cnt, c_cnt = int((ca & ~cc).sum()), int((~ca & cc).sum())
        p_mc = binomtest(min(b_cnt, c_cnt), b_cnt + c_cnt, 0.5).pvalue if (b_cnt + c_cnt) > 0 else 1.0
        pair_rows.append({
            "himpunan": label, "model_A": a, "model_B": c,
            "selisih_acc": pa["acc"] - pc["acc"],
            "acc_ci_low": np.percentile(da, 2.5), "acc_ci_high": np.percentile(da, 97.5),
            "p_boot_acc": boot_p(da),
            "A_benar_B_salah": b_cnt, "A_salah_B_benar": c_cnt, "p_mcnemar": p_mc,
            "selisih_f1": pa["f1"] - pc["f1"],
            "f1_ci_low": np.percentile(df, 2.5), "f1_ci_high": np.percentile(df, 97.5),
            "f1_ci995_low": np.percentile(df, 0.25), "f1_ci995_high": np.percentile(df, 99.75),
            "p_boot_f1": boot_p(df),
        })
    pairs = pd.DataFrame(pair_rows)
    for col in ["p_boot_acc", "p_mcnemar", "p_boot_f1"]:
        pairs[col + "_holm"] = holm(pairs[col].values)
    pairs["signifikan_f1_holm"] = pairs["p_boot_f1_holm"] < ALPHA
    pairs["signifikan_acc_mcnemar_holm"] = pairs["p_mcnemar_holm"] < ALPHA
    return per_model, pairs

res = {
    "bersih": analyze(~is_flag, "bersih"),
    "asli": analyze(np.ones(len(ref), dtype=bool), "asli"),
}

## 4. Hasil per model (titik estimasi dan CI 95%)

In [5]:
pm = pd.concat([res["bersih"][0], res["asli"][0]], ignore_index=True)
pm.to_csv(os.path.join(OUT_DIR, "per_model.csv"), index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
pm.round(4)

,himpunan,model,n_gambar,acc,acc_ci_low,acc_ci_high,prec,prec_ci_low,prec_ci_high,rec,rec_ci_low,rec_ci_high,f1,f1_ci_low,f1_ci_high
0,bersih,ViT-B/16,1058,0.8025,0.7779,0.8261,0.8343,0.8075,0.8557,0.8089,0.7831,0.8340,0.8133,0.7825,0.8343
1,bersih,Improved Swin,1058,0.7921,0.7675,0.8157,0.8198,0.7959,0.8420,0.7994,0.7738,0.8240,0.8023,0.7726,0.8235
2,bersih,MobileNetV3 Large,1058,0.7769,0.7514,0.8025,0.7888,0.7612,0.8142,0.7873,0.7603,0.8128,0.7822,0.7498,0.8040
3,bersih,ResNet50,1058,0.7618,0.7353,0.7873,0.7786,0.7506,0.8036,0.7659,0.7381,0.7921,0.7622,0.7287,0.7843
4,bersih,DenseNet121,1058,0.6323,0.6030,0.6607,0.6595,0.6268,0.6903,0.6412,0.6100,0.6710,0.6427,0.6061,0.6665
5,asli,ViT-B/16,1438,0.8470,0.8282,0.8658,0.8682,0.8499,0.8854,0.8452,0.8249,0.8652,0.8511,0.8288,0.8680
6,asli,Improved Swin,1438,0.8352,0.8157,0.8540,0.8512,0.8312,0.8700,0.8291,0.8075,0.8500,0.8347,0.8105,0.8526
7,asli,MobileNetV3 Large,1438,0.8234,0.8039,0.8428,0.8337,0.8130,0.8535,0.8209,0.7991,0.8418,0.8232,0.7989,0.8411
8,asli,ResNet50,1438,0.8122,0.7914,0.8324,0.8180,0.7953,0.8399,0.8063,0.7845,0.8281,0.8058,0.7799,0.8251
9,asli,DenseNet121,1438,0.6912,0.6676,0.7149,0.7107,0.6857,0.7355,0.6922,0.6674,0.7162,0.6956,0.6673,0.7159


## 5. Pemeriksaan kecocokan dengan naskah (jalankan sebelum memakai angka)

In [6]:
NASKAH_ACC_ASLI = {"ViT-B/16": 0.8713, "Improved Swin": 0.8352, "MobileNetV3 Large": 0.8234,
                   "ResNet50": 0.8122, "DenseNet121": 0.6912}
chk = res["asli"][0][["model", "acc"]].copy()
chk["acc_naskah"] = chk["model"].map(NASKAH_ACC_ASLI)
chk["beda"] = (chk["acc"] - chk["acc_naskah"]).round(4)
chk["cocok"] = chk["beda"].abs() < 0.0005
chk.round(4)

,model,acc,acc_naskah,beda,cocok
0,ViT-B/16,0.8470,0.8713,-0.0243,False
1,Improved Swin,0.8352,0.8352,-0.0000,True
2,MobileNetV3 Large,0.8234,0.8234,-0.0000,True
3,ResNet50,0.8122,0.8122,0.0000,True
4,DenseNet121,0.6912,0.6912,0.0000,True


## 6. Selisih berpasangan: test set bersih (hasil utama)

In [7]:
cols = ["model_A", "model_B",
        "selisih_f1", "f1_ci_low", "f1_ci_high", "f1_ci995_low", "f1_ci995_high", "p_boot_f1_holm", "signifikan_f1_holm",
        "selisih_acc", "acc_ci_low", "acc_ci_high", "p_mcnemar_holm", "signifikan_acc_mcnemar_holm"]
pr_bersih = res["bersih"][1]
pr_bersih[cols].round(4)

,model_A,model_B,selisih_f1,f1_ci_low,f1_ci_high,f1_ci995_low,f1_ci995_high,p_boot_f1_holm,signifikan_f1_holm,selisih_acc,acc_ci_low,acc_ci_high,p_mcnemar_holm,signifikan_acc_mcnemar_holm
0,ViT-B/16,Improved Swin,0.0110,-0.0130,0.0333,-0.0232,0.0432,0.3528,False,0.0104,-0.0113,0.0321,0.5846,False
1,ViT-B/16,MobileNetV3 Large,0.0311,0.0057,0.0566,-0.0060,0.0686,0.0688,False,0.0255,0.0028,0.0482,0.1457,False
2,ViT-B/16,ResNet50,0.0510,0.0241,0.0789,0.0127,0.0904,0.0012,True,0.0406,0.0161,0.0652,0.0123,True
3,ViT-B/16,DenseNet121,0.1706,0.1417,0.2035,0.1272,0.2168,0.0000,True,0.1701,0.1418,0.1985,0.0000,True
4,Improved Swin,MobileNetV3 Large,0.0201,-0.0039,0.0456,-0.0167,0.0583,0.3186,False,0.0151,-0.0066,0.0359,0.5846,False
5,Improved Swin,ResNet50,0.0401,0.0151,0.0675,0.0031,0.0801,0.0140,True,0.0302,0.0076,0.0539,0.0700,False
6,Improved Swin,DenseNet121,0.1596,0.1309,0.1938,0.1174,0.2092,0.0000,True,0.1597,0.1323,0.1881,0.0000,True
7,MobileNetV3 Large,ResNet50,0.0199,-0.0061,0.0470,-0.0175,0.0568,0.3186,False,0.0151,-0.0095,0.0397,0.5846,False
8,MobileNetV3 Large,DenseNet121,0.1395,0.1128,0.1706,0.0995,0.1831,0.0000,True,0.1446,0.1172,0.1730,0.0000,True
9,ResNet50,DenseNet121,0.1196,0.0893,0.1529,0.0749,0.1669,0.0000,True,0.1295,0.1002,0.1597,0.0000,True


## 7. Selisih berpasangan: test set asli (pembanding)

In [8]:
pr_asli = res["asli"][1]
pr_asli[cols].round(4)

,model_A,model_B,selisih_f1,f1_ci_low,f1_ci_high,f1_ci995_low,f1_ci995_high,p_boot_f1_holm,signifikan_f1_holm,selisih_acc,acc_ci_low,acc_ci_high,p_mcnemar_holm,signifikan_acc_mcnemar_holm
0,ViT-B/16,Improved Swin,0.0164,-0.0025,0.0358,-0.0114,0.0448,0.2334,False,0.0118,-0.0049,0.0285,0.5329,False
1,ViT-B/16,MobileNetV3 Large,0.0278,0.0084,0.0483,-0.0012,0.0574,0.0304,True,0.0236,0.0063,0.0417,0.0582,False
2,ViT-B/16,ResNet50,0.0453,0.0239,0.0679,0.0139,0.0782,0.0024,True,0.0348,0.0160,0.0542,0.0032,True
3,ViT-B/16,DenseNet121,0.1555,0.1312,0.1817,0.1204,0.1920,0.0000,True,0.1558,0.1328,0.1787,0.0000,True
4,Improved Swin,MobileNetV3 Large,0.0114,-0.0069,0.0298,-0.0151,0.0386,0.2334,False,0.0118,-0.0042,0.0278,0.5329,False
5,Improved Swin,ResNet50,0.0288,0.0091,0.0496,0.0003,0.0584,0.0240,True,0.0229,0.0049,0.0410,0.0590,False
6,Improved Swin,DenseNet121,0.1390,0.1147,0.1649,0.1042,0.1771,0.0000,True,0.1439,0.1217,0.1662,0.0000,True
7,MobileNetV3 Large,ResNet50,0.0174,-0.0019,0.0369,-0.0096,0.0459,0.2334,False,0.0111,-0.0076,0.0292,0.5329,False
8,MobileNetV3 Large,DenseNet121,0.1276,0.1040,0.1526,0.0931,0.1635,0.0000,True,0.1321,0.1099,0.1551,0.0000,True
9,ResNet50,DenseNet121,0.1102,0.0846,0.1366,0.0739,0.1470,0.0000,True,0.1210,0.0967,0.1453,0.0000,True


In [9]:
def table3(label):
    pm_l = res[label][0]
    pr_l = res[label][1]
    A = pd.DataFrame({
        "Model": pm_l["model"],
        "Accuracy [95% CI]": [f"{r.acc:.4f} [{r.acc_ci_low:.4f}, {r.acc_ci_high:.4f}]" for r in pm_l.itertuples()],
        "Macro precision [95% CI]": [f"{r.prec:.4f} [{r.prec_ci_low:.4f}, {r.prec_ci_high:.4f}]" for r in pm_l.itertuples()],
        "Macro recall [95% CI]": [f"{r.rec:.4f} [{r.rec_ci_low:.4f}, {r.rec_ci_high:.4f}]" for r in pm_l.itertuples()],
        "Macro F1 [95% CI]": [f"{r.f1:.4f} [{r.f1_ci_low:.4f}, {r.f1_ci_high:.4f}]" for r in pm_l.itertuples()],
    })
    Bt = pd.DataFrame({
        "Pasangan": [f"{r.model_A} vs {r.model_B}" for r in pr_l.itertuples()],
        "Selisih macro F1 [95% CI]": [f"{r.selisih_f1:+.4f} [{r.f1_ci_low:+.4f}, {r.f1_ci_high:+.4f}]" for r in pr_l.itertuples()],
        "p Holm (F1)": [fmt_p(p) for p in pr_l["p_boot_f1_holm"]],
        "Signifikan (F1)": np.where(pr_l["signifikan_f1_holm"], "ya", "tidak"),
        "Selisih akurasi [95% CI]": [f"{r.selisih_acc:+.4f} [{r.acc_ci_low:+.4f}, {r.acc_ci_high:+.4f}]" for r in pr_l.itertuples()],
        "p McNemar Holm": [fmt_p(p) for p in pr_l["p_mcnemar_holm"]],
    })
    return A, Bt

for label in ("bersih", "asli"):
    A, Bt = table3(label)
    A.to_csv(os.path.join(OUT_DIR, f"table3A_{label}.csv"), index=False)
    Bt.to_csv(os.path.join(OUT_DIR, f"table3B_{label}.csv"), index=False)

A, Bt = table3("bersih")
display(A); display(Bt)

,Model,Accuracy [95% CI],Macro precision [95% CI],Macro recall [95% CI],Macro F1 [95% CI]
0,ViT-B/16,"0.8025 [0.7779, 0.8261]","0.8343 [0.8075, 0.8557]","0.8089 [0.7831, 0.8340]","0.8133 [0.7825, 0.8343]"
1,Improved Swin,"0.7921 [0.7675, 0.8157]","0.8198 [0.7959, 0.8420]","0.7994 [0.7738, 0.8240]","0.8023 [0.7726, 0.8235]"
2,MobileNetV3 Large,"0.7769 [0.7514, 0.8025]","0.7888 [0.7612, 0.8142]","0.7873 [0.7603, 0.8128]","0.7822 [0.7498, 0.8040]"
3,ResNet50,"0.7618 [0.7353, 0.7873]","0.7786 [0.7506, 0.8036]","0.7659 [0.7381, 0.7921]","0.7622 [0.7287, 0.7843]"
4,DenseNet121,"0.6323 [0.6030, 0.6607]","0.6595 [0.6268, 0.6903]","0.6412 [0.6100, 0.6710]","0.6427 [0.6061, 0.6665]"


,Pasangan,Selisih macro F1 [95% CI],p Holm (F1),Signifikan (F1),Selisih akurasi [95% CI],p McNemar Holm
0,ViT-B/16 vs Improved Swin,"+0.0110 [-0.0130, +0.0333]",0.3528,tidak,"+0.0104 [-0.0113, +0.0321]",0.5846
1,ViT-B/16 vs MobileNetV3 Large,"+0.0311 [+0.0057, +0.0566]",0.0688,tidak,"+0.0255 [+0.0028, +0.0482]",0.1457
2,ViT-B/16 vs ResNet50,"+0.0510 [+0.0241, +0.0789]",0.0012,ya,"+0.0406 [+0.0161, +0.0652]",0.0123
3,ViT-B/16 vs DenseNet121,"+0.1706 [+0.1417, +0.2035]",< 0.0001,ya,"+0.1701 [+0.1418, +0.1985]",< 0.0001
4,Improved Swin vs MobileNetV3 Large,"+0.0201 [-0.0039, +0.0456]",0.3186,tidak,"+0.0151 [-0.0066, +0.0359]",0.5846
5,Improved Swin vs ResNet50,"+0.0401 [+0.0151, +0.0675]",0.0140,ya,"+0.0302 [+0.0076, +0.0539]",0.0700
6,Improved Swin vs DenseNet121,"+0.1596 [+0.1309, +0.1938]",< 0.0001,ya,"+0.1597 [+0.1323, +0.1881]",< 0.0001
7,MobileNetV3 Large vs ResNet50,"+0.0199 [-0.0061, +0.0470]",0.3186,tidak,"+0.0151 [-0.0095, +0.0397]",0.5846
8,MobileNetV3 Large vs DenseNet121,"+0.1395 [+0.1128, +0.1706]",< 0.0001,ya,"+0.1446 [+0.1172, +0.1730]",< 0.0001
9,ResNet50 vs DenseNet121,"+0.1196 [+0.0893, +0.1529]",< 0.0001,ya,"+0.1295 [+0.1002, +0.1597]",< 0.0001


## 9. Simpan semua hasil

In [10]:
pd.concat([res["bersih"][1], res["asli"][1]], ignore_index=True).to_csv(
    os.path.join(OUT_DIR, "pairwise.csv"), index=False)
print("Tersimpan di", OUT_DIR, "->", sorted(os.listdir(OUT_DIR)))

Tersimpan di stats_report -> ['pairwise.csv', 'per_model.csv', 'table3A_asli.csv', 'table3A_bersih.csv', 'table3B_asli.csv', 'table3B_bersih.csv']
